In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

customers_bronze = spark.table(
    "workspace.fashion_bronze.customers_ingested"
)

print("Bronze customer records:", customers_bronze.count())

customers_bronze.printSchema()

Bronze customer records: 100000
root
 |-- city: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- preferred_language: string (nullable = true)
 |-- acquisition_channel: string (nullable = true)
 |-- pincode: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- age_group: string (nullable = true)



In [0]:
customers_bronze.select(
    F.count("*").alias("total_records"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.sum(F.col("customer_id").isNull().cast("int")).alias("null_customer_ids"),
    F.countDistinct("city").alias("unique_cities"),
    F.countDistinct("gender").alias("gender_categories"),
    F.min("age").alias("minimum_age"),
    F.max("age").alias("maximum_age")
).show()

+-------------+----------------+-----------------+-------------+-----------------+-----------+-----------+
|total_records|unique_customers|null_customer_ids|unique_cities|gender_categories|minimum_age|maximum_age|
+-------------+----------------+-----------------+-------------+-----------------+-----------+-----------+
|       100000|          100000|                0|            4|                3|         23|         56|
+-------------+----------------+-----------------+-------------+-----------------+-----------+-----------+



In [0]:
null_counts = customers_bronze.select([
    F.sum(
        F.col(column).isNull().cast("int")
    ).alias(column)
    for column in customers_bronze.columns
])

display(null_counts)

city,customer_id,first_name,last_name,customer_name,email,registration_date,date_of_birth,gender,preferred_language,acquisition_channel,pincode,age,age_group
0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
categorical_columns = [
    "city",
    "gender",
    "preferred_language",
    "acquisition_channel",
    "age_group"
]

for column in categorical_columns:
    print(f"\nDistinct values in {column}:")
    customers_bronze.select(column) \
        .distinct() \
        .orderBy(column) \
        .show(truncate=False)


Distinct values in city:
+---------+
|city     |
+---------+
|Bengaluru|
|Delhi    |
|Mumbai   |
|Pune     |
+---------+


Distinct values in gender:
+------+
|gender|
+------+
|Female|
|Male  |
|Other |
+------+


Distinct values in preferred_language:
+------------------+
|preferred_language|
+------------------+
|English           |
|Hindi             |
|Kannada           |
|Marathi           |
+------------------+


Distinct values in acquisition_channel:
+-------------------+
|acquisition_channel|
+-------------------+
|Direct             |
|Email              |
|Facebook Ads       |
|Google Ads         |
|Instagram Ads      |
|Organic Search     |
|Organic Social     |
|WhatsApp           |
+-------------------+


Distinct values in age_group:
+---------+
|age_group|
+---------+
|18-24    |
|25-34    |
|35-44    |
|45-54    |
|55+      |
+---------+



In [0]:
customers_silver = (
    customers_bronze
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("gender", F.initcap(F.trim(F.col("gender"))))
    .withColumn(
        "preferred_language",
        F.initcap(F.trim(F.col("preferred_language")))
    )
    .withColumn(
        "acquisition_channel",
        F.trim(F.col("acquisition_channel"))
    )
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn(
        "customer_name",
        F.trim(F.regexp_replace(F.col("customer_name"), r"\s+", " "))
    )
)

In [0]:
customers_silver.select(
    F.count("*").alias("total_records"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.sum(F.col("email").isNull().cast("int")).alias("null_emails"),
    F.sum(
        (F.col("email") != F.lower(F.trim(F.col("email")))).cast("int")
    ).alias("non_standard_emails"),
    F.sum(
        (F.col("customer_name") != F.trim(F.col("customer_name"))).cast("int")
    ).alias("names_with_outer_spaces")
).show()

+-------------+----------------+-----------+-------------------+-----------------------+
|total_records|unique_customers|null_emails|non_standard_emails|names_with_outer_spaces|
+-------------+----------------+-----------+-------------------+-----------------------+
|       100000|          100000|          0|                  0|                      0|
+-------------+----------------+-----------+-------------------+-----------------------+



In [0]:
print(
    "Invalid ages:",
    customers_silver.filter(
        (F.col("age") < 18) | (F.col("age") > 100)
    ).count()
)

print(
    "Invalid email formats:",
    customers_silver.filter(
        ~F.col("email").rlike(r"^[A-Za-z0-9._%+-]+@example\.com$")
    ).count()
)

print(
    "Invalid registration dates:",
    customers_silver.filter(
        (F.col("registration_date") < F.lit("2023-01-01")) |
        (F.col("registration_date") > F.current_date())
    ).count()
)

Invalid ages: 0
Invalid email formats: 0
Invalid registration dates: 0


In [0]:
customers_silver = customers_silver.withColumn(
    "expected_age_group",
    F.when(F.col("age") <= 24, "18-24")
    .when(F.col("age") <= 34, "25-34")
    .when(F.col("age") <= 44, "35-44")
    .when(F.col("age") <= 54, "45-54")
    .otherwise("55+")
)

print(
    "Age-group mismatches:",
    customers_silver.filter(
        F.col("age_group") != F.col("expected_age_group")
    ).count()
)

customers_silver = customers_silver.drop("expected_age_group")

Age-group mismatches: 0


In [0]:
duplicate_customer_ids = (
    customers_silver
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate customer IDs:", duplicate_customer_ids.count())

Duplicate customer IDs: 0


In [0]:
customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.fashion_silver.customers")

In [0]:
customers_silver_saved = spark.table(
    "workspace.fashion_silver.customers"
)

print("Silver customer records:", customers_silver_saved.count())

customers_silver_saved.printSchema()

customers_silver_saved.show(5, truncate=False)

Silver customer records: 100000
root
 |-- city: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- preferred_language: string (nullable = true)
 |-- acquisition_channel: string (nullable = true)
 |-- pincode: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- age_group: string (nullable = true)

+---------+-----------+----------+---------+-------------+------------------------------+-----------------+-------------+------+------------------+-------------------+-------+---+---------+
|city     |customer_id|first_name|last_name|customer_name|email                         |registration_date|date_of_birth|gender|preferred_language|acquisition_channel|pincode|age|ag